# Suite2p — Batch runner (shared settings, one `db` per session)

Load a settings preset once, then call `run_s2p(db=db, settings=settings)` for each session.

## `settings` vs `db`

| | `settings` | `db` |
|--|--|--|
| **Contains** | all processing parameters (nested dict) | the dataset: paths, planes, channels, ROIs, … |
| **Scope** | shared across sessions, loaded from a file | one per session |
| **Per-session tweaks** | optional nested `settings` overrides per session | — |

The settings preset is the JSON written by the pipeline notebook (or a `settings.npy` from any suite2p results folder). It is deep-merged onto `default_settings()`, so keys missing from the file fall back to the defaults.

**Reference:** Stringer, Ki et al. 2026 · https://github.com/MouseLand/suite2p

In [ ]:
# %pip install suite2p

import suite2p
from suite2p import run_s2p, default_settings, default_db
import numpy as np
import json
import time
import traceback
from pathlib import Path
from datetime import datetime

import copy

def deep_update(base, new):
    """Recursively merge `new` into `base` (nested dicts are merged, not replaced).
    run_s2p() only does a shallow merge, so a partial group such as
    {"registration": {"nonrigid": False}} would otherwise wipe the other
    registration defaults."""
    for k, v in new.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            deep_update(base[k], v)
        else:
            base[k] = v
    return base

def find_unknown_keys(d, ref, prefix=""):
    """Keys in `d` that do not exist in the reference dict (typos / removed keys)."""
    bad = []
    for k, v in d.items():
        if k not in ref:
            bad.append(prefix + k)
        elif isinstance(v, dict) and isinstance(ref[k], dict):
            bad += find_unknown_keys(v, ref[k], prefix + k + ".")
    return bad

def to_jsonable(v):
    if isinstance(v, dict):            return {k: to_jsonable(x) for k, x in v.items()}
    if isinstance(v, (list, tuple)):   return [to_jsonable(x) for x in v]
    if isinstance(v, np.ndarray):      return v.tolist()
    if isinstance(v, np.generic):      return v.item()
    if isinstance(v, Path):            return str(v)
    return v

def load_settings_file(path):
    """Load a settings file: a .json written by these notebooks, or a settings.npy
    written by suite2p into any suite2p/ or suite2p/planeN/ folder."""
    path = Path(path)
    assert path.exists(), f"Settings file not found: {path}"
    if path.suffix == ".json":
        with open(path) as f:
            return json.load(f)
    if path.suffix == ".npy":
        return np.load(path, allow_pickle=True).item()
    raise ValueError(f"Unsupported format: {path.suffix}")

print(f"suite2p {suite2p.__version__}")

## 1. Load the shared settings

In [ ]:
SETTINGS_FILE = r"/path/to/Dendrites_iGluSNFR4s.json"   # or settings.npy from any suite2p results folder

settings_loaded = load_settings_file(SETTINGS_FILE)

# Deep-merge onto the defaults (so keys added in newer suite2p versions are present)
settings = deep_update(default_settings(), settings_loaded)

unknown = find_unknown_keys(settings, default_settings())
if unknown:
    print(f"⚠️  Keys not in default_settings() (typos or pre-1.0 keys?): {unknown}")

print(f"Loaded settings from: {Path(SETTINGS_FILE).name}")
print("\nKey parameters:")
for label, val in [
    ("fs",                          settings["fs"]),
    ("tau",                         settings["tau"]),
    ("diameter",                    settings["diameter"]),
    ("registration.nonrigid",       settings["registration"]["nonrigid"]),
    ("registration.batch_size",     settings["registration"]["batch_size"]),
    ("detection.algorithm",         settings["detection"]["algorithm"]),
    ("detection.threshold_scaling", settings["detection"]["threshold_scaling"]),
    ("extraction.neuropil_coefficient", settings["extraction"]["neuropil_coefficient"]),
]:
    print(f"  {label:34s} = {val}")

## 2. Define the sessions

Each session has a `db` (dataset description — at minimum `data_path` and `save_path0`) and, optionally, a `settings` dict with **nested overrides for that session only**, e.g. `{"fs": 15.0}` or `{"detection": {"threshold_scaling": 2.0}}`.

In [ ]:
sessions = [
    {
        "db": {
            "data_path": [
                "/data/raw/mouse01/2026-01-15/run1",
                "/data/raw/mouse01/2026-01-15/run2",
                "/data/raw/mouse01/2026-01-15/run3",
            ],
            "save_path0": "/data/processed/mouse01/2026-01-15",
            "fast_disk":  "/scratch",
        },
    },
    {
        "db": {
            "data_path": [
                "/data/raw/mouse01/2026-01-16/run1",
                "/data/raw/mouse01/2026-01-16/run2",
            ],
            "save_path0": "/data/processed/mouse01/2026-01-16",
            "fast_disk":  "/scratch",
        },
    },
    {
        "db": {
            "data_path": [
                "/data/raw/mouse02/2026-01-20/run1",
                "/data/raw/mouse02/2026-01-20/run2",
            ],
            "save_path0": "/data/processed/mouse02/2026-01-20",
            "nplanes": 4,                                  # dataset property → db
        },
        "settings": {"fs": 15.0},                          # ← per-session processing override: 15 Hz
    },
    # add more sessions here …
]

# Fail early on typos
for s in sessions:
    bad = find_unknown_keys(s["db"], default_db()) + find_unknown_keys(s.get("settings", {}), default_settings())
    assert not bad, f"Unknown keys in session {s['db']['data_path'][0]}: {bad}"

## 3. Run the pipeline

In [ ]:
results = []

for i, sess in enumerate(sessions):
    db = {**default_db(), **sess["db"]}
    s  = deep_update(copy.deepcopy(settings), sess.get("settings", {}))
    save_dir = Path(db["save_path0"]) / db["save_folder"]
    label = db["data_path"][0]   # use the data path as a human-readable label
    print(f"\n{'─'*60}")
    print(f"[{i+1}/{len(sessions)}]  {label}")

    t0 = time.time()
    try:
        run_s2p(db=db, settings=s)
        elapsed = time.time() - t0

        # Record the exact settings + db used next to the results
        with open(save_dir / "settings_used.json", "w") as f:
            json.dump(to_jsonable(s), f, indent=2)
        with open(save_dir / "db_used.json", "w") as f:
            json.dump(to_jsonable(db), f, indent=2)

        results.append({"db": db, "status": "OK", "elapsed": elapsed, "save": save_dir})
        print(f"✓  {elapsed/60:.1f} min  →  {save_dir}")

    except Exception as exc:
        elapsed = time.time() - t0
        results.append({"db": db, "status": "FAILED", "elapsed": elapsed, "error": str(exc)})
        print(f"❌  FAILED after {elapsed/60:.1f} min")
        traceback.print_exc()   # print and continue — don't abort the batch

print(f"\n{'═'*60}")
ok     = [r for r in results if r["status"] == "OK"]
failed = [r for r in results if r["status"] != "OK"]
print(f"Done: {len(ok)}/{len(results)} succeeded, {len(failed)} failed")

## 4. Summary & QC

In [ ]:
import matplotlib.pyplot as plt

summary = []
for r in ok:
    for plane_dir in sorted(Path(r["save"]).glob("plane*")):
        iscell_path = plane_dir / "iscell.npy"
        if not iscell_path.exists():
            continue
        iscell = np.load(iscell_path, allow_pickle=True)
        summary.append({
            "label":   Path(r["db"]["data_path"][0]).name + f" / {plane_dir.name}",
            "n_cell":  int(iscell[:, 0].sum()),
            "n_total": len(iscell),
        })

if summary:
    fig, ax = plt.subplots(figsize=(max(5, len(summary) * 1.2), 4))
    x = range(len(summary))
    n_cell    = [s["n_cell"]  for s in summary]
    n_noncell = [s["n_total"] - s["n_cell"] for s in summary]
    ax.bar(x, n_cell,    label="cells",     color="steelblue")
    ax.bar(x, n_noncell, bottom=n_cell, label="non-cells", color="lightgray")
    ax.set_xticks(list(x))
    ax.set_xticklabels([s["label"] for s in summary], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("ROI count")
    ax.set_title("Detected cells per session")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"\n{'Session':<45} {'Cells':>7} {'Total':>7} {'% cells':>8}")
    print("-" * 72)
    for s in summary:
        pct = 100 * s["n_cell"] / max(s["n_total"], 1)
        print(f"{s['label']:<45} {s['n_cell']:>7} {s['n_total']:>7} {pct:>7.1f}%")

In [ ]:
# Save run log
log = {
    "run_date":      datetime.now().isoformat(),
    "suite2p_ver":   suite2p.__version__,
    "settings_file": str(SETTINGS_FILE),
    "results":       [{"data_path": r["db"]["data_path"],
                       "save_path": r["db"]["save_path0"],
                       "status":    r["status"],
                       "elapsed_min": round(r["elapsed"]/60, 2),
                       **({"error": r["error"]} if "error" in r else {})
                       } for r in results],
}
log_path = Path(sessions[0]["db"]["save_path0"]).parent / "batch_run_log.json"
with open(log_path, "w") as f:
    json.dump(log, f, indent=2, default=str)
print(f"Run log → {log_path}")

---
## Tips

**Skip already-processed sessions:**
```python
sessions = [s for s in sessions
            if not (Path(s["db"]["save_path0"]) / "suite2p" / "plane0" / "iscell.npy").exists()]
```

**Retry only failed sessions:**
```python
failed_paths = {r["db"]["data_path"][0] for r in results if r["status"] != "OK"}
sessions = [s for s in sessions if s["db"]["data_path"][0] in failed_paths]
```

**Where does each kind of override go?**

| Scenario | Where |
|---|---|
| Multi-plane | `db`: `"nplanes": 4` |
| Two-channel | `db`: `"nchannels": 2` |
| Different frame rate | session `settings`: `{"fs": 15.0}` |
| Different indicator | session `settings`: `{"tau": 0.3}` |
| Noisier session | session `settings`: `{"detection": {"threshold_scaling": 2.0}}` |
| Align to the second channel | session `settings`: `{"registration": {"align_by_chan2": True}}` |
| Skip registration (data already registered) | session `settings`: `{"run": {"do_registration": 0}}` — check in the 1.0 docs how to point `db["data_path"]` at the existing binary |